# 🔬 SkinVision AI — Model Training (Google Colab)

**Trains EfficientNetB0 on your skin disease dataset using free GPU.**

### Steps:
1. Run Cell 1 — connect Google Drive
2. Run Cell 2 — upload your zip files OR point to Drive folder
3. Run Cell 3 — install deps
4. Run Cell 4 — organize dataset
5. Run Cell 5 — TRAIN (~20 min on GPU)
6. Run Cell 6 — download `skinvision_best.keras`
7. Copy downloaded file → `backend/models/skinvision_best.keras`
8. `docker restart skinvision-backend` — real AI active!


In [ ]:
# ── Cell 1: Check GPU + Mount Google Drive ──────────────────────────────────
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
# ── Cell 2: Upload zip files ─────────────────────────────────────────────────
# OPTION A: Upload directly (drag & drop in Files panel on left)
# OPTION B: Copy from Google Drive (change the path below)

import os, zipfile, shutil
from pathlib import Path

WORK_DIR = Path('/content/skinvision')
DATA_DIR = WORK_DIR / 'data' / 'train'
for cls in ['melanoma', 'non_melanoma', 'acne', 'healthy']:
    (DATA_DIR / cls).mkdir(parents=True, exist_ok=True)

# ─── CHANGE THESE PATHS to where your zip files are in Drive ────────────────
ZIP_FILES = [
    # '/content/drive/MyDrive/archive (1).zip',   # DermNet
    # '/content/drive/MyDrive/archive (2).zip',   # HAM10000 / other
]

# Or upload directly:
from google.colab import files
print('Upload your zip files (or skip if using Drive paths above)')
# uploaded = files.upload()  # uncomment to upload

print('Data dir ready:', DATA_DIR)

In [ ]:
# ── Cell 3: Install deps ─────────────────────────────────────────────────────
!pip install -q scipy scikit-learn
print('Done!')

In [ ]:
# ── Cell 4: Extract zips + Auto-organize dataset ─────────────────────────────
import zipfile, shutil
from pathlib import Path
from collections import defaultdict

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

KEYWORD_MAP = {
    'melanoma':    ['melanoma', 'mel_', '_mel', 'superficial', 'nodular', 'lentigo', 'acral'],
    'non_melanoma':['basal', 'bcc', 'squamous', 'scc', 'akiec', 'actinic', 'keratosis',
                    'carcinoma', 'wart', 'molluscum', 'viral', 'vascular', 'seborrheic',
                    'seborrhoeic', 'benign_keratosis', 'bkl'],
    'acne':        ['eczema', 'atopic', 'dermatitis', 'psoriasis', 'lichen',
                    'rosacea', 'acne', 'pimple', 'inflammatory'],
    'healthy':     ['nevi', 'nv', 'naevi', 'nevus', 'mole', 'benign_mole',
                    'healthy', 'normal', 'melanocytic'],
}

def classify_folder(name):
    n = name.lower()
    for cls, kws in KEYWORD_MAP.items():
        if any(k in n for k in kws):
            return cls
    return None

def safe_copy(src, dst_dir):
    dst = dst_dir / src.name
    if dst.exists():
        dst = dst_dir / f'{src.parent.name}_{src.name}'
    if not dst.exists():
        shutil.copy2(src, dst)
        return True
    return False

def organize_from_root(root, max_healthy=2000):
    root = Path(root)
    counts = defaultdict(int)
    healthy_count = 0
    for folder in root.rglob('*'):
        if not folder.is_dir(): continue
        cls = classify_folder(folder.name)
        if not cls: continue
        for img in folder.iterdir():
            if img.suffix.lower() not in IMAGE_EXTS: continue
            if cls == 'healthy' and healthy_count >= max_healthy: continue
            if safe_copy(img, DATA_DIR / cls):
                counts[cls] += 1
                if cls == 'healthy': healthy_count += 1
    return counts

# Extract and organize each zip
EXTRACT_DIR = Path('/content/extracted')
EXTRACT_DIR.mkdir(exist_ok=True)

all_zips = list(Path('/content').glob('*.zip'))
for zp in ZIP_FILES:
    if Path(zp).exists():
        all_zips.append(Path(zp))

total = defaultdict(int)
for zp in all_zips:
    print(f'Extracting {zp.name}...')
    dest = EXTRACT_DIR / zp.stem
    with zipfile.ZipFile(zp, 'r') as z:
        z.extractall(dest)
    counts = organize_from_root(dest)
    print(f'  Copied: {dict(counts)}')
    for k, v in counts.items():
        total[k] += v

print('\n=== Final Dataset ===')
for cls in ['melanoma','non_melanoma','acne','healthy']:
    n = len(list((DATA_DIR/cls).glob('*')))
    status = 'OK' if n >= 100 else ('LOW' if n >= 20 else 'TOO FEW')
    print(f'  {cls:<20} {n:>5} images [{status}]')

In [ ]:
# ── Cell 5: TRAIN MODEL ──────────────────────────────────────────────────────
import json, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path

IMG_SIZE        = (224, 224)
BATCH_SIZE      = 32          # GPU can handle 32
EPOCHS_FROZEN   = 15
EPOCHS_FINETUNE = 25
MODELS_DIR      = WORK_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# Scan classes
classes = sorted([d.name for d in DATA_DIR.iterdir()
                  if d.is_dir() and len(list(d.glob('*'))) >= 20])
print(f'Training {len(classes)} classes: {classes}')

# Class weights to handle imbalance
counts = {c: len(list((DATA_DIR/c).glob('*'))) for c in classes}
total_imgs = sum(counts.values())
class_weight = {i: total_imgs / (len(classes) * counts[c]) for i, c in enumerate(classes)}
print('Class weights:', {classes[i]: round(w,2) for i,w in class_weight.items()})

# Data generators with augmentation
train_gen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2,
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, zoom_range=0.15, brightness_range=[0.8,1.2],
)
val_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = train_gen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    classes=classes, subset='training', seed=42)
val_data = val_gen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    classes=classes, subset='validation', seed=42)

# Build model
def build_model(n):
    base = EfficientNetB0(weights='imagenet', include_top=False,
                          input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.45)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n, activation='softmax')(x)
    return keras.Model(base.input, out), base

model, base = build_model(len(classes))

callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(MODELS_DIR/'skinvision_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
]

# Phase 1: frozen base
print('\n=== Phase 1: Training head (base frozen) ===')
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, validation_data=val_data, epochs=EPOCHS_FROZEN,
          class_weight=class_weight, callbacks=callbacks)

# Phase 2: fine-tune top 40 layers
print('\n=== Phase 2: Fine-tuning top 40 layers ===')
for layer in base.layers[-40:]:
    layer.trainable = True
model.compile(optimizer=keras.optimizers.Adam(5e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, validation_data=val_data,
          epochs=EPOCHS_FROZEN + EPOCHS_FINETUNE,
          initial_epoch=EPOCHS_FROZEN,
          class_weight=class_weight, callbacks=callbacks)

# Save final + classes
model.save(str(MODELS_DIR/'skinvision_final.keras'))
with open(str(MODELS_DIR/'classes.json'), 'w') as f:
    json.dump(classes, f)

print('\n=== Training Complete! ===')
print(f'Best model: {MODELS_DIR}/skinvision_best.keras')
print(f'Classes: {classes}')

In [ ]:
# ── Cell 6: Download trained model ───────────────────────────────────────────
from google.colab import files
import os

model_path = str(WORK_DIR / 'models' / 'skinvision_best.keras')
classes_path = str(WORK_DIR / 'models' / 'classes.json')

print('Downloading skinvision_best.keras...')
files.download(model_path)

print('Downloading classes.json...')
files.download(classes_path)

print()
print('=== NEXT STEPS ===')
print('1. Copy skinvision_best.keras → D:\\skinvision (1)\\skinvision\\backend\\models\\')
print('2. Copy classes.json          → same folder')
print('3. Run: docker restart skinvision-backend')
print('4. Check: Invoke-RestMethod http://localhost:8000/health')
print('   model_loaded: true  ← Real AI active!')